# Abstract

This file answers the following questions:

- What can you say about transitions from one product to another?
- What's the most popular path?


In [1]:
from working_with_db import *
from printmd import *

conn = connect_to_local_postgresql(db_name='postgres', user='jet', password='brains')

number_of_products_df = execute_query_to_dataframe(conn, read_sql_file_to_string(
    'general_sql/number_of_products_per_user.sql'))

common_df = execute_query_to_dataframe(conn, read_sql_file_to_string('transition_paths/transition_paths_common.sql'))
transition_paths_df = execute_query_to_dataframe(conn, read_sql_file_to_string('transition_paths/transition_paths.sql'))

# General information
___


## Data context

1. The calculation doesn't take parallel usage of products. \
If a customer has two or more parallel products, then false transitions are present in the result table.
For example: A customer has two X licenses. He downgraded one license to D, and the second license to B. In the result, we can see the transition path from D to B.

2. The source data is not complete, so there are users with incomplete history.\
For example, clients having the "Upgrade" as the first event.\
These "Upgrades" were excluded from the final transition table.\
These are customers who purchased a license before 2018 (before the start of the data period in the source).

In [2]:
printmd(f"\nAnd {common_df['share_customers_with_upgrade'][0]}% of customers have a license with type 'Upgrade'.")
printmd('\nThe table shows the number of users and the percentage with one, two, etc. products')
printdf(number_of_products_df)


And 12.18% of customers have a license with type 'Upgrade'.


The table shows the number of users and the percentage with one, two, etc. products

|   number_of_products |   number_of_users |   proportion_of_users |
|---------------------:|------------------:|----------------------:|
|                    1 |              8858 |                 84.67 |
|                    2 |              1380 |                 13.19 |
|                    3 |               189 |                  1.81 |
|                    4 |                28 |                  0.27 |
|                    5 |                 5 |                  0.05 |
|                    6 |                 1 |                  0.01 |
|                    8 |                 1 |                  0.01 |

## Observation

- Most customers use one product.

# The most popular transition paths
___
## Calculation



In [4]:
printmd('\nThis table identifies the top 10 most common paths users take when transitioning between products.')
printdf(transition_paths_df)


This table identifies the top 10 most common paths users take when transitioning between products.

| previous_product   | upgrade_product   |   number_of_transitions |
|:-------------------|:------------------|------------------------:|
| H                  | I                 |                     253 |
| A                  | X                 |                     218 |
| B                  | X                 |                     105 |
| I                  | I                 |                      77 |
| C                  | X                 |                      69 |
| D                  | X                 |                      69 |
| H                  | X                 |                      46 |
| X                  | A                 |                      29 |
| I                  | X                 |                      28 |
| F                  | X                 |                      26 |

## Observations

Most often, customers upgrade to products X and I.

I can assume that:

- product I is the dotUltimate Pack because it can only be upgraded from product H.
- product H it is Rider or ReSharper. DotNET-developers upgrade from one dotNET tool to Pack.
- product X is the All Products Pack because users upgrade to it from any product.



If this statement is true, then transitions from product X and I to any other are a downgrade.


# Ideas for further research

1. Create TOP transitions for customers from different groups (individual use and organizations)\
From the available data we can make an assumption about the type of user based on the license price.
These groups may have different transition patterns.
2. Check the downgrade from products X and I to the previous or new products.\
For example: The customer purchased product D, after some time upgraded to X, but then downgraded to B.\
This may indicate a wrong choice of product at the start.